# FILE LOADER 

Usa la piattaforma di ATLAS per caricare file dentro la repo di progetto su S3

In [2]:
import digitalhub as dh
dh.get_current_profile()


'__default'

In [3]:
PROJECT_NAME = "i3-indire"
project = dh.get_or_create_project(PROJECT_NAME)

## Leggi i PDF in una tabella

Una riga per PDF, incluse le sottocartelle. `content` contiene i binari Python (`bytes`), senza conversione in testo o Base64. I percorsi sono relativi alla directory di lavoro del notebook.

Tutti i binari vengono tenuti in RAM: per la cartella attuale servono circa 928 MB, oltre alla memoria usata dal notebook. Usa `recursive=False` per leggere solo la directory indicata.


In [4]:
from pathlib import Path
import hashlib
import pandas as pd


def read_pdfs(directory, recursive=True):
    directory = Path(directory).expanduser().resolve()
    if not directory.is_dir():
        raise NotADirectoryError(f"Directory not found: {directory}")

    paths = directory.rglob("*") if recursive else directory.iterdir()
    rows = []
    for path in sorted(paths):
        if not path.is_file() or path.suffix.lower() != ".pdf":
            continue
        content = path.read_bytes()
        
        rows.append({
            "file_name": path.name,
            "local_path": str(path),
            "relative_path": path.relative_to(directory).as_posix(),
            "mime_type": "application/pdf",
            "size_bytes": len(content),
            "sha256": hashlib.sha256(content).hexdigest(),
            "content": content,
        })

    return pd.DataFrame(rows, columns=[
        "file_name", "local_path", "relative_path", "mime_type",
        "size_bytes", "sha256", "content",
    ])


PDF_DIRECTORY = Path("./file_to_load")
pdf_df = read_pdfs(PDF_DIRECTORY,recursive=False)
print(f"PDFs loaded: {len(pdf_df)}; total bytes: {pdf_df['size_bytes'].sum():,}")
# Preview without displaying binary content.
pdf_df.drop(columns="content").head()


PDFs loaded: 5; total bytes: 5,789,058


,file_name,local_path,relative_path,mime_type,size_bytes,sha256
0,Modulo1_Lezione1-1_Bibliografia.pdf,/home/cappelle/indire/file_to_load/Modulo1_Lez...,Modulo1_Lezione1-1_Bibliografia.pdf,application/pdf,474693,af2422afd5173735bd21efa70bba0753eb5954f014b797...
1,Modulo1_Lezione1-1_Caso1.pdf,/home/cappelle/indire/file_to_load/Modulo1_Lez...,Modulo1_Lezione1-1_Caso1.pdf,application/pdf,324733,a7ef1e0edd4a53e249d6efe93fdb6a7d762e0f9ecbe77c...
2,Modulo1_Lezione1-1_Caso2.pdf,/home/cappelle/indire/file_to_load/Modulo1_Lez...,Modulo1_Lezione1-1_Caso2.pdf,application/pdf,1227045,28966478ab484f821a1c6ce317679fab48e9292edede77...
3,Modulo1_Lezione1-1_Guida.pdf,/home/cappelle/indire/file_to_load/Modulo1_Lez...,Modulo1_Lezione1-1_Guida.pdf,application/pdf,309772,f2a0fc1e55869daacb333b912e9e3bca95fea3913f2b9a...
4,Modulo1_Lezione1-1_Materiale.pdf,/home/cappelle/indire/file_to_load/Modulo1_Lez...,Modulo1_Lezione1-1_Materiale.pdf,application/pdf,3452815,035e65fd768bbf98bed3bc5cfd451543d59bfa2ae32fe3...


## Caricamento successivo con DigitalHub Core

usa la funzione core per trasformare in markdown il pdf. il risultato viene messo su S3

`project.log_artifact(kind="artifact", source=..., path=...)` carica un file binario nello storage S3 e lo registra nel progetto. La funzione seguente scrive temporaneamente i byte della tabella su disco perché `source` richiede un percorso locale. Mantiene le sottocartelle nella destinazione S3.

Configura `S3_PREFIX` con il bucket e il prefisso desiderati, usando il profilo DigitalHub già configurato. La definizione della funzione non esegue upload; decommenta la chiamata solo quando vuoi caricare i file. Ogni esecuzione richiama l'upload per tutte le righe, senza deduplicazione remota.

Riferimento: [DigitalHub SDK — log_artifact](https://scc-digitalhub.github.io/sdk-docs/0.7/api/entities/project/entity/).


In [17]:
func = project.new_function(
        name="export-md-func", 
        kind="python",
        code_src="./pdf2md_handler.py",
        handler="pdf3mdFunction",
        python_version="PYTHON3_10",
        requirements=[
                "pymupdf",
                "pymupdf4llm",
        ]
        
)

In [18]:
from tempfile import TemporaryDirectory





def upload_pdfs_to_s3(table, project, s3_prefix):
    if not s3_prefix.startswith("s3://") or not s3_prefix[5:].split("/", 1)[0]:
        raise ValueError("Specify a destination such as s3://bucket/prefix")

    results = []
    with TemporaryDirectory(prefix="pdf-upload-") as tmp:
        for row in table.itertuples(index=False):
            destination = f"{s3_prefix.rstrip('/')}/{row.relative_path}"
            # Use a stable, distinct name for PDFs with the same filename in different folders.
            identifier = hashlib.sha256(
                row.relative_path.encode("utf-8")
            ).hexdigest()[:24]
            source = Path(tmp) / Path(row.file_name).name
            source.write_bytes(row.content)
            try:
                artifact = project.log_artifact(
                    name=f"{row.file_name}-{identifier}",
                    kind="artifact",
                    source=str(source),
                    path=destination,
                )
                run = func.run(action="job", parameters={"artifact": artifact.key})
                print(run.result("message"))
                
            finally:
                source.unlink(missing_ok=True)
            results.append({
                "relative_path": row.relative_path,
                "s3_uri": destination,
                "sha256": row.sha256,
                "artifact_name": artifact.name,
            })

    return pd.DataFrame(results, columns=["relative_path", "s3_uri", "sha256", ])


S3_PREFIX = "s3://test-datalake/i3-indire/core"
upload_df = upload_pdfs_to_s3(pdf_df, project, S3_PREFIX)
print(f"PDFs uploaded: {len(upload_df)}; total bytes: {pdf_df['size_bytes'].sum():,}")

None
None
None
None
None
PDFs uploaded: 5; total bytes: 5,789,058


In [8]:
print(upload_df.head())



                         relative_path  \
0  Modulo1_Lezione1-1_Bibliografia.pdf   
1         Modulo1_Lezione1-1_Caso1.pdf   
2         Modulo1_Lezione1-1_Caso2.pdf   
3         Modulo1_Lezione1-1_Guida.pdf   
4     Modulo1_Lezione1-1_Materiale.pdf   

                                              s3_uri  \
0  s3://test-datalake/i3-indire/core/Modulo1_Lezi...   
1  s3://test-datalake/i3-indire/core/Modulo1_Lezi...   
2  s3://test-datalake/i3-indire/core/Modulo1_Lezi...   
3  s3://test-datalake/i3-indire/core/Modulo1_Lezi...   
4  s3://test-datalake/i3-indire/core/Modulo1_Lezi...   

                                              sha256  
0  af2422afd5173735bd21efa70bba0753eb5954f014b797...  
1  a7ef1e0edd4a53e249d6efe93fdb6a7d762e0f9ecbe77c...  
2  28966478ab484f821a1c6ce317679fab48e9292edede77...  
3  f2a0fc1e55869daacb333b912e9e3bca95fea3913f2b9a...  
4  035e65fd768bbf98bed3bc5cfd451543d59bfa2ae32fe3...  


In [5]:
func = project.new_function(
        name="export-md-func", 
        kind="python",
        code_src="./pdf2md_handler.py",
        handler="pdf3mdFunction",
        python_version="PYTHON3_10",
)

In [10]:
for row in pdf_df.itertuples(index=False):
    print(f"Exporting {row.relative_path} to markdown...")
    run = func.run(action="job", parameters={"s3Path": row.local_path, "content": row.content})
    print(run.result("message"))
    

Exporting Modulo1_Lezione1-1_Bibliografia.pdf to markdown...
None
Exporting Modulo1_Lezione1-1_Caso1.pdf to markdown...
None
Exporting Modulo1_Lezione1-1_Caso2.pdf to markdown...
None
Exporting Modulo1_Lezione1-1_Guida.pdf to markdown...
None
Exporting Modulo1_Lezione1-1_Materiale.pdf to markdown...
None
